# Machine Learning Final Project
# Image Classification with CIFAR-10

**Task:** Train a Machine Learning/Deep Learning model on the CIFAR-10 dataset for image classification.

**Framework:** PyTorch

**Author:** [Your Name] - [Your Student ID]

**Date:** September 2026

---


# 1. Preparation

## 1.1 Set Up Virtual Environment

```bash
# Verify Python installation
python --version

# Set up virtual environment using conda
conda create -n ml_project python=3.11
conda activate ml_project

# Install necessary libraries
pip install torch torchvision matplotlib numpy pandas scikit-learn
```


## 1.2 Import Libraries


In [ ]:
import torch
from torch import nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import random
import os

# Check versions
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


---
## 1.3 Understand the Data

### Load the dataset


In [ ]:
# Load CIFAR-10 with basic transform for exploration
basic_transform = transforms.Compose([
    transforms.ToTensor()
])

train_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=basic_transform
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=basic_transform
)

print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")


### 1.3.1 What are the feature columns?


In [ ]:
# CIFAR-10 contains 32x32 RGB images
# Features = pixel values (3 channels x 32 height x 32 width)
img, label = train_data[0]
print(f"Image tensor shape: {img.shape}")
print(f"Interpretation: {img.shape[0]} color channels x {img.shape[1]} height x {img.shape[2]} width")
print(f"Total features per image: {img.shape[0] * img.shape[1] * img.shape[2]} pixel values")
print(f"\nLabel: {label} ({train_data.classes[label]})")


### 1.3.2 What feature columns are numerical?


In [ ]:
# All features are numerical (pixel values)
print("All features are NUMERICAL (pixel intensity values)")
print(f"Data type: {img.dtype}")
print(f"Value range: [{img.min():.4f}, {img.max():.4f}]")
print(f"After ToTensor(), values are normalized to [0, 1]")


### 1.3.3 Min, Max, Mean, Standard Deviation of numerical features


In [ ]:
# Statistics using the raw data (numpy array)
data_array = train_data.data  # Shape: (50000, 32, 32, 3), uint8 [0-255]

print("=== Raw pixel statistics (0-255 range) ===")
print(f"Min: {data_array.min()}")
print(f"Max: {data_array.max()}")
print(f"Overall Mean: {data_array.mean():.2f}")
print(f"Overall Std: {data_array.std():.2f}")

print(f"\n=== Per-channel statistics (normalized to 0-1) ===")
# Normalize to 0-1 for per-channel stats
data_normalized = data_array / 255.0
for i, channel in enumerate(['Red', 'Green', 'Blue']):
    ch_data = data_normalized[:, :, :, i]
    print(f"{channel}: Mean={ch_data.mean():.4f}, Std={ch_data.std():.4f}, "
          f"Min={ch_data.min():.4f}, Max={ch_data.max():.4f}")


### 1.3.4 What feature columns are categorical?


In [ ]:
# For image data, there are NO categorical features
# All pixel values are continuous numerical values
print("There are NO categorical feature columns in CIFAR-10.")
print("All features (pixel values) are continuous numerical values.")


### 1.3.5 How many categories does each categorical feature column have?


In [ ]:
# Not applicable for image data
# However, the TARGET column (labels) is categorical with 10 classes
print("No categorical FEATURES exist.")
print(f"\nThe TARGET variable has {len(train_data.classes)} categories:")
for idx, name in enumerate(train_data.classes):
    print(f"  {idx}: {name}")


### 1.3.6 Value counts per category


In [ ]:
# Count samples per class
train_labels = [label for _, label in train_data]
test_labels = [label for _, label in test_data]

train_counts = Counter(train_labels)
test_counts = Counter(test_labels)

print("=== Training Set - Samples per class ===")
for idx in sorted(train_counts.keys()):
    print(f"  {train_data.classes[idx]:>12}: {train_counts[idx]} samples")

print(f"\n=== Test Set - Samples per class ===")
for idx in sorted(test_counts.keys()):
    print(f"  {test_data.classes[idx]:>12}: {test_counts[idx]} samples")

# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar([train_data.classes[i] for i in sorted(train_counts.keys())],
            [train_counts[i] for i in sorted(train_counts.keys())],
            color='steelblue')
axes[0].set_title('Training Set Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar([test_data.classes[i] for i in sorted(test_counts.keys())],
            [test_counts[i] for i in sorted(test_counts.keys())],
            color='coral')
axes[1].set_title('Test Set Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


### 1.3.7 Is there any missing data?


In [ ]:
# Check for missing data
print("CIFAR-10 is a well-curated built-in dataset.")
print(f"Training data shape: {train_data.data.shape}")
print(f"Any NaN values: {np.isnan(train_data.data.astype(float)).any()}")
print(f"Any null/zero images: {(train_data.data.sum(axis=(1,2,3)) == 0).any()}")
print(f"\n=> There is NO missing data in CIFAR-10.")


### Visualize sample images


In [ ]:
# Visualize random samples from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Sample Images from Each Class", fontsize=16)

for class_idx in range(10):
    row = class_idx // 5
    col = class_idx % 5

    # Find a sample of this class
    indices = [i for i, (_, label) in enumerate(train_data) if label == class_idx]
    sample_idx = random.choice(indices[:100])  # Pick from first 100 to be fast
    img, label = train_data[sample_idx]

    axes[row, col].imshow(img.permute(1, 2, 0))
    axes[row, col].set_title(f"{train_data.classes[class_idx]}")
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()


---
## 1.4 Understand Your Task


In [ ]:
print("=" * 50)
print("TASK UNDERSTANDING")
print("=" * 50)

print(f"\n1. What will you classify?")
print(f"   => We will classify IMAGES into one of 10 categories")

print(f"\n2. What is the target column?")
print(f"   => The target is the 'label' (integer 0-9)")
print(f"   => Mapping: {dict(zip(range(10), train_data.classes))}")

print(f"\n3. How many classes does the dataset have?")
print(f"   => {len(train_data.classes)} classes")

print(f"\n4. Task type: Multi-class Classification")
print(f"   => Input: 32x32x3 RGB image")
print(f"   => Output: Probability distribution over 10 classes")


---
## 1.5 Brainstorm Data Preprocessing Steps

### 1.5.1 What should you do to handle missing data?
CIFAR-10 has no missing data, so no handling is needed.

### 1.5.2 What scale/normalization methods should you use?
- **Normalize** pixel values using per-channel mean and std
- Mean: (0.4914, 0.4822, 0.4465)
- Std: (0.2470, 0.2435, 0.2616)

### 1.5.3 What encoding methods for categorical columns?
- No categorical features in image data

### 1.5.4 What encoding methods for the target column?
- Labels are already integers (0-9)
- `CrossEntropyLoss` handles this automatically

### 1.5.5 How to split the dataset?
- CIFAR-10 comes pre-split: 50,000 train + 10,000 test


In [ ]:
# Define transforms
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# Training transform with data augmentation
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),          # Random crop with padding
    transforms.RandomHorizontalFlip(p=0.5),        # Random horizontal flip
    transforms.ToTensor(),                          # Convert to tensor [0,1]
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)  # Normalize
])

# Test transform (no augmentation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

print("Training transform:")
print(train_transform)
print(f"\nTest transform:")
print(test_transform)


In [ ]:
# Apply transforms and create DataLoaders
train_data.transform = train_transform
test_data.transform = test_transform

BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

train_dataloader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

test_dataloader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

# Verify
image_batch, label_batch = next(iter(train_dataloader))
print(f"Image batch shape: {image_batch.shape}")
print(f"Label batch shape: {label_batch.shape}")
print(f"Number of training batches: {len(train_dataloader)}")
print(f"Number of test batches: {len(test_dataloader)}")


---
## 1.6 Select a Model

### Which model should you use?
We will use a **CNN (Convolutional Neural Network)** based on a VGG-like architecture.

**Why CNN?**
- Images have spatial structure (nearby pixels are related)
- CNNs use convolutional layers to detect local patterns (edges, textures, shapes)
- Much more parameter-efficient than fully-connected networks for image data

### Which hyperparameters?
| Hyperparameter | Value | Rationale |
|---|---|---|
| Hidden units | 32 | Balance between capacity and speed |
| Kernel size | 3x3 | Standard for capturing local patterns |
| Learning rate | 0.001 | Default for Adam optimizer |
| Batch size | 32 | Fits in memory, provides stable gradients |
| Epochs | 10 | Enough to see convergence |
| Optimizer | Adam | Adaptive learning rate, fast convergence |
| Loss function | CrossEntropyLoss | Standard for multi-class classification |


---
# 2. Build the Model


In [ ]:
class CIFAR10_CNN(nn.Module):
    """
    VGG-like CNN for CIFAR-10 classification.

    Architecture:
    - Conv Block 1: 2x Conv2d(3→hidden) + ReLU + MaxPool
    - Conv Block 2: 2x Conv2d(hidden→hidden) + ReLU + MaxPool
    - Classifier: Flatten → Linear → 10 classes
    """
    def __init__(self, input_channels: int, hidden_units: int, output_classes: int):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_channels, out_channels=hidden_units,
                      kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units,
                      kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 32x32 -> 16x16
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units * 2,
                      kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(hidden_units * 2),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units * 2, out_channels=hidden_units * 2,
                      kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(hidden_units * 2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 2 * 8 * 8, out_features=256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features=256, out_features=output_classes)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)
        return x


In [ ]:
# Instantiate model
torch.manual_seed(42)

model = CIFAR10_CNN(
    input_channels=3,
    hidden_units=32,
    output_classes=len(train_data.classes)
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# Test forward pass
dummy_input = torch.randn(1, 3, 32, 32).to(device)
output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output (logits): {output}")


---
# 3. Training

## 3.1 Define loss function and optimizer


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

print(f"Loss function: {loss_fn}")
print(f"Optimizer: {optimizer}")


## 3.2 Define training and testing step functions


In [ ]:
def train_step(model, dataloader, loss_fn, optimizer, device):
    """Train model for one epoch."""
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Forward pass
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate accuracy
        y_pred_class = torch.argmax(y_pred, dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y)

    # Average loss and accuracy
    train_loss /= len(dataloader)
    train_acc /= len(dataloader)
    return train_loss, train_acc


def test_step(model, dataloader, loss_fn, device):
    """Evaluate model on test set."""
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            test_loss += loss.item()

            y_pred_class = torch.argmax(y_pred, dim=1)
            test_acc += (y_pred_class == y).sum().item() / len(y)

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)
    return test_loss, test_acc


## 3.3 Training loop


In [ ]:
def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs, device):
    """Full training loop with progress tracking."""
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_step(model, train_dataloader, loss_fn, optimizer, device)
        test_loss, test_acc = test_step(model, test_dataloader, loss_fn, device)

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        print(f"Epoch {epoch+1:>2}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return results


In [ ]:
# Start training!
torch.manual_seed(42)
torch.cuda.manual_seed(42)

EPOCHS = 10

print(f"Training on {device} for {EPOCHS} epochs...")
print("=" * 80)

model_results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=EPOCHS,
    device=device
)

print("=" * 80)
print(f"Final Test Accuracy: {model_results['test_acc'][-1]:.4f}")


---
# 4. Evaluation & Results

## 4.1 Training curves


In [ ]:
def plot_training_curves(results):
    """Plot training and test loss/accuracy curves."""
    epochs_range = range(1, len(results["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(epochs_range, results["train_loss"], label="Train Loss", marker='o')
    axes[0].plot(epochs_range, results["test_loss"], label="Test Loss", marker='s')
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(epochs_range, results["train_acc"], label="Train Accuracy", marker='o')
    axes[1].plot(epochs_range, results["test_acc"], label="Test Accuracy", marker='s')
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("Training Curves", fontsize=14)
    plt.tight_layout()
    plt.show()

plot_training_curves(model_results)


## 4.2 Predictions on test samples


In [ ]:
def make_predictions(model, data, device):
    """Make predictions on a list of samples."""
    pred_probs = []
    model.eval()
    with torch.inference_mode():
        for sample in data:
            sample = torch.unsqueeze(sample, dim=0).to(device)
            pred_logit = model(sample)
            pred_prob = torch.softmax(pred_logit.squeeze(), dim=0)
            pred_probs.append(pred_prob.cpu())
    return torch.stack(pred_probs)


# Get random test samples
random.seed(42)
test_samples = []
test_labels_sample = []
for sample, label in random.sample(list(test_data), k=9):
    test_samples.append(sample)
    test_labels_sample.append(label)

# Make predictions
pred_probs = make_predictions(model=model, data=test_samples, device=device)
pred_classes = pred_probs.argmax(dim=1)

# Visualize
fig = plt.figure(figsize=(12, 12))
nrows, ncols = 3, 3
for i, sample in enumerate(test_samples):
    plt.subplot(nrows, ncols, i + 1)

    # Denormalize for display
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    img_display = sample * std + mean
    img_display = torch.clamp(img_display, 0, 1)

    plt.imshow(img_display.permute(1, 2, 0))

    pred_label = train_data.classes[pred_classes[i]]
    truth_label = train_data.classes[test_labels_sample[i]]

    color = "green" if pred_label == truth_label else "red"
    plt.title(f"Pred: {pred_label}\nTrue: {truth_label}",
              fontsize=10, color=color)
    plt.axis("off")

plt.suptitle("Predictions vs Ground Truth", fontsize=14)
plt.tight_layout()
plt.show()


## 4.3 Confusion Matrix


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def get_all_predictions(model, dataloader, device):
    """Get predictions for entire dataset."""
    all_preds = []
    all_labels = []

    model.eval()
    with torch.inference_mode():
        for X, y in dataloader:
            X = X.to(device)
            y_pred = model(X)
            pred_class = torch.argmax(y_pred, dim=1)
            all_preds.extend(pred_class.cpu().numpy())
            all_labels.extend(y.numpy())

    return np.array(all_preds), np.array(all_labels)

# Get all predictions
all_preds, all_labels = get_all_predictions(model, test_dataloader, device)

# Classification report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(all_labels, all_preds,
                            target_names=train_data.classes))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=train_data.classes,
            yticklabels=train_data.classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


---
# 5. Bonus 🎉

## 5.1 Experiment with different models

We will compare our custom CNN with a simpler model and a pretrained model.


### Model 2: Simple CNN (fewer layers)


In [ ]:
class SimpleCNN(nn.Module):
    """A simpler CNN with just one conv block."""
    def __init__(self, input_channels, output_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 32->16
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16->8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, output_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Train Simple CNN
torch.manual_seed(42)
simple_model = SimpleCNN(input_channels=3, output_classes=10).to(device)
simple_optimizer = torch.optim.Adam(simple_model.parameters(), lr=0.001)

print(f"Simple CNN - Parameters: {sum(p.numel() for p in simple_model.parameters()):,}")
print("Training Simple CNN...")
simple_results = train(
    model=simple_model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=simple_optimizer,
    loss_fn=loss_fn,
    epochs=EPOCHS,
    device=device
)


### Model 3: Deeper CNN


In [ ]:
class DeeperCNN(nn.Module):
    """A deeper CNN with 3 conv blocks."""
    def __init__(self, input_channels, hidden_units, output_classes):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(input_channels, hidden_units, 3, padding=1),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, 3, padding=1),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 32->16
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units * 2, 3, padding=1),
            nn.BatchNorm2d(hidden_units * 2),
            nn.ReLU(),
            nn.Conv2d(hidden_units * 2, hidden_units * 2, 3, padding=1),
            nn.BatchNorm2d(hidden_units * 2),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 16->8
        )
        self.conv_block_3 = nn.Sequential(
            nn.Conv2d(hidden_units * 2, hidden_units * 4, 3, padding=1),
            nn.BatchNorm2d(hidden_units * 4),
            nn.ReLU(),
            nn.Conv2d(hidden_units * 4, hidden_units * 4, 3, padding=1),
            nn.BatchNorm2d(hidden_units * 4),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 8->4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_units * 4 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, output_classes)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.conv_block_3(x)
        x = self.classifier(x)
        return x

# Train Deeper CNN
torch.manual_seed(42)
deeper_model = DeeperCNN(input_channels=3, hidden_units=32, output_classes=10).to(device)
deeper_optimizer = torch.optim.Adam(deeper_model.parameters(), lr=0.001)

print(f"Deeper CNN - Parameters: {sum(p.numel() for p in deeper_model.parameters()):,}")
print("Training Deeper CNN...")
deeper_results = train(
    model=deeper_model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=deeper_optimizer,
    loss_fn=loss_fn,
    epochs=EPOCHS,
    device=device
)


## 5.2 Compare all models


In [ ]:
def compare_models(results_dict):
    """Compare training results of multiple models."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = ['steelblue', 'coral', 'seagreen', 'purple']

    for i, (name, results) in enumerate(results_dict.items()):
        epochs_range = range(1, len(results["test_loss"]) + 1)
        color = colors[i % len(colors)]

        axes[0].plot(epochs_range, results["test_loss"],
                     label=f"{name}", marker='o', color=color)
        axes[1].plot(epochs_range, results["test_acc"],
                     label=f"{name}", marker='o', color=color)

    axes[0].set_title("Test Loss Comparison")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title("Test Accuracy Comparison")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("Model Comparison", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Print summary table
    print("\n" + "=" * 60)
    print(f"{'Model':<20} {'Final Test Acc':>15} {'Final Test Loss':>15}")
    print("=" * 60)
    for name, results in results_dict.items():
        print(f"{name:<20} {results['test_acc'][-1]:>15.4f} {results['test_loss'][-1]:>15.4f}")

compare_models({
    "VGG-like CNN": model_results,
    "Simple CNN": simple_results,
    "Deeper CNN": deeper_results
})


## 5.3 Experiment with different preprocessing

Compare results with and without data augmentation.


In [ ]:
# Experiment: Train without data augmentation
no_aug_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

# Create new dataset without augmentation
train_data_no_aug = datasets.CIFAR10(root="data", train=True, download=True,
                                      transform=no_aug_transform)
train_loader_no_aug = DataLoader(train_data_no_aug, batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=NUM_WORKERS)

# Train same model architecture without augmentation
torch.manual_seed(42)
model_no_aug = CIFAR10_CNN(input_channels=3, hidden_units=32, output_classes=10).to(device)
optimizer_no_aug = torch.optim.Adam(model_no_aug.parameters(), lr=0.001)

print("Training VGG-like CNN WITHOUT data augmentation...")
no_aug_results = train(
    model=model_no_aug,
    train_dataloader=train_loader_no_aug,
    test_dataloader=test_dataloader,
    optimizer=optimizer_no_aug,
    loss_fn=loss_fn,
    epochs=EPOCHS,
    device=device
)

# Compare
compare_models({
    "With Augmentation": model_results,
    "Without Augmentation": no_aug_results,
})


---
# 6. Conclusion

## Summary of Results

Based on our experiments:

1. **VGG-like CNN** with data augmentation provides a good balance of performance and training speed
2. **Data augmentation** (RandomCrop, RandomHorizontalFlip) improves generalization
3. **Deeper architectures** can improve accuracy but require more training time
4. **Batch Normalization** and **Dropout** help prevent overfitting

## Key Learnings

- Image classification with CNNs is effective for CIFAR-10
- Data preprocessing (normalization, augmentation) significantly impacts results
- Model architecture choice affects both accuracy and training time
- Hyperparameter tuning (learning rate, batch size) is important for convergence

## Future Work

- Try transfer learning with pretrained models (ResNet, VGG16)
- Implement learning rate scheduling
- Try more aggressive data augmentation
- Increase training epochs for better convergence


---
# 7. Save the Model


In [ ]:
# Save the best model
model_save_path = "models/"
os.makedirs(model_save_path, exist_ok=True)

torch.save(model.state_dict(),
           os.path.join(model_save_path, "cifar10_vgg_cnn.pth"))
print(f"Model saved to {model_save_path}cifar10_vgg_cnn.pth")
